In [1]:
%matplotlib inline

In [2]:
#Import your libraries here

import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path
from datetime import datetime

In [3]:
#Import your modules here
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from utils.ad_parsing import ad_parsing_utils as adp
from utils.fetch_data import (
                                ScrapeSelection,
                                collect_listing_urls_for_routes_parallel_streaming,
                                download_and_parse_listing_batch_streaming, 
                                load_valid_deal_types,
                                load_valid_geo_paths,
                                load_valid_property_types
                                )

print(PROJECT_ROOT)

D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_1


In [4]:
RAW_HTML_DIR = PROJECT_ROOT / "data" / "raw_listing_html"
RAW_HTML_DIR.mkdir(parents=True, exist_ok=True)

for file_path in RAW_HTML_DIR.glob("*.html"):
    file_path.unlink()

TAXONOMY_DIR = PROJECT_ROOT / "data" / "taxonomy"
DATA_DIR = PROJECT_ROOT / "data"

print(TAXONOMY_DIR)

# ============================================================
# Run mode
# ============================================================

RUN_MODE = "resume"
# Options:
# "new"    = create a fresh independent run
# "resume" = continue a previous interrupted run


# Use only when RUN_MODE = "resume"
RESUME_RUN_ID = "20260608_081646"  # replace with the actual old RUN_ID


if RUN_MODE == "new":
    RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
elif RUN_MODE == "resume":
    RUN_ID = RESUME_RUN_ID
else:
    raise ValueError("RUN_MODE must be either 'new' or 'resume'")

ROUTE_DISCOVERY_DIR = (
    PROJECT_ROOT
    / "data"
    / "route_discovery_runs"
    / f"sales_full_{RUN_ID}"
)

PARSED_OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "parsed_sales_runs"
    / f"parsed_sales_full_{RUN_ID}"
)

if RUN_MODE == "new":
    ROUTE_DISCOVERY_DIR.mkdir(parents=True, exist_ok=False)
    PARSED_OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

elif RUN_MODE == "resume":
    if not ROUTE_DISCOVERY_DIR.exists():
        raise FileNotFoundError(f"Cannot resume. Missing route folder: {ROUTE_DISCOVERY_DIR}")

    PARSED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_MODE:", RUN_MODE)
print("RUN_ID:", RUN_ID)
print("Route discovery dir:", ROUTE_DISCOVERY_DIR)
print("Parsed output dir:", PARSED_OUTPUT_DIR)

D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_1\data\taxonomy
RUN_MODE: resume
RUN_ID: 20260608_081646
Route discovery dir: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_1\data\route_discovery_runs\sales_full_20260608_081646
Parsed output dir: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_1\data\parsed_sales_runs\parsed_sales_full_20260608_081646


## Call the generated url and obtain a list of urls with ads to the related query

In [ ]:
valid_deal_types = load_valid_deal_types(TAXONOMY_DIR / "valid_deal_types.csv")
valid_geo_paths = load_valid_geo_paths(TAXONOMY_DIR / "valid_geo_paths.csv")
valid_property_types = load_valid_property_types(TAXONOMY_DIR / "valid_property_types.csv")


all_sales_geo_paths = [
    geo_path
    for deal_type, geo_path in valid_geo_paths
    if deal_type == "prodazhbi"
]

selection = ScrapeSelection(
    deal_types=["prodazhbi"],
    geo_paths=all_sales_geo_paths,
    property_types=list(valid_property_types),
)

route_urls = selection.build_urls(
    valid_deal_types,
    valid_geo_paths,
    valid_property_types,
)

print("Geo paths:", len(all_sales_geo_paths))
print("Property types:", len(valid_property_types))
print("Route URLs:", len(route_urls))
print(route_urls[:10])

run_summary = collect_listing_urls_for_routes_parallel_streaming(
    route_urls=route_urls,
    max_workers=24,
    max_pages=200,
    delay_seconds=0.6,
    checkpoint_dir=ROUTE_DISCOVERY_DIR,
    progress_every=100,
)

run_summary

Geo paths: 4375
Property types: 22
Route URLs: 96250
['https://www.imot.bg/obiavi/prodazhbi/oblast-burgas/s-malina/mezonet', 'https://www.imot.bg/obiavi/prodazhbi/oblast-burgas/s-malina/mnogostaen', 'https://www.imot.bg/obiavi/prodazhbi/oblast-burgas/s-malina/kashta', 'https://www.imot.bg/obiavi/prodazhbi/oblast-burgas/s-malina/chetiristaen', 'https://www.imot.bg/obiavi/prodazhbi/oblast-burgas/s-malina/atelie-tavan', 'https://www.imot.bg/obiavi/prodazhbi/oblast-burgas/s-malina/vila', 'https://www.imot.bg/obiavi/prodazhbi/oblast-burgas/s-malina/myasto', 'https://www.imot.bg/obiavi/prodazhbi/oblast-burgas/s-malina/dvustaen', 'https://www.imot.bg/obiavi/prodazhbi/oblast-burgas/s-malina/magazin', 'https://www.imot.bg/obiavi/prodazhbi/oblast-burgas/s-malina/garazh-parkomyasto']
Total routes: 96250
Already completed: 72737
Pending routes: 23513
[72738/96250] routes completed | new this run=1 | route listings=0 | pages=1 | stop=no_listing_urls | url=https://www.imot.bg/obiavi/prodazhbi/oblast

In [ ]:
listing_urls_raw_path = ROUTE_DISCOVERY_DIR / "listing_urls_raw.csv"
listing_urls_final_path = ROUTE_DISCOVERY_DIR / "listing_urls_unique.csv"

listing_urls_df = pd.read_csv(listing_urls_raw_path)

unique_listing_urls_df = (
    listing_urls_df[["listing_url"]]
    .drop_duplicates()
    .sort_values("listing_url")
    .reset_index(drop=True)
)

unique_listing_urls_df.to_csv(listing_urls_final_path, index=False)

print("Run ID:", RUN_ID)
print("Raw listing rows:", len(listing_urls_df))
print("Unique listing URLs:", len(unique_listing_urls_df))
print("Saved to:", listing_urls_final_path)

## Read the list of URLS from the generated query

In [ ]:
# ============================================================
# Load discovered listing URLs from current route crawl run
# Then download + parse listings using streaming pipeline
# ============================================================

# Use the ROUTE_DISCOVERY_DIR already created in Cell 3.
# Do NOT redefine it here.

RAW_LISTING_URLS_CSV = ROUTE_DISCOVERY_DIR / "listing_urls_unique.csv"

PARSED_OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "parsed_sales_runs"
    / f"parsed_sales_full_{RUN_ID}"
)

PARSED_OUTPUT_DIR.mkdir(parents=True, exist_ok=False)


# ------------------------------------------------------------
# 1. Load listing URLs from route crawl output
# ------------------------------------------------------------

df_urls = pd.read_csv(RAW_LISTING_URLS_CSV)

listing_urls = (
    df_urls["listing_url"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

print("RUN_ID:", RUN_ID)
print("Route discovery dir:", ROUTE_DISCOVERY_DIR)
print("Parsed output dir:", PARSED_OUTPUT_DIR)
print("Loaded unique listing URLs:", len(listing_urls))
print(listing_urls[:5])


# ------------------------------------------------------------
# 2. Download + parse listings using streaming/resumable method
# ------------------------------------------------------------

download_parse_summary = download_and_parse_listing_batch_streaming(
    listing_urls=listing_urls,
    output_dir=PARSED_OUTPUT_DIR,
    max_workers=24,
    limit=None,          # None = process all
    chunk_size=3000,     # keeps futures bounded
    progress_every=300,
)

download_parse_summary


# ------------------------------------------------------------
# 3. Optional: inspect parsed output
# ------------------------------------------------------------

parsed_csv = PARSED_OUTPUT_DIR / "parsed_listings.csv"
manifest_csv = PARSED_OUTPUT_DIR / "download_manifest.csv"

print("Download manifest:", manifest_csv)
print("Parsed listings:", parsed_csv)

if parsed_csv.exists():
    df_parsed = pd.read_csv(parsed_csv)
    print("Parsed rows:", len(df_parsed))
    display(df_parsed.head())
else:
    print("No parsed listings CSV created yet.")

In [ ]:
csv_path = PARSED_OUTPUT_DIR / "parsed_listings.csv"
parquet_path = PARSED_OUTPUT_DIR / "parsed_listings.parquet"

df = pd.read_csv(csv_path)

df.to_parquet(
    parquet_path,
    index=False,
    engine="pyarrow",
    compression="snappy",
)

print("Saved:", parquet_path)
print("Rows:", len(df))
print("Columns:", len(df.columns))